# Exercises for session 5 (solution)

> **Note:**
> 
> Please commit every time you solve one of the exercises. An example commit message
> could be `"Solution to question 1"`. Feel free to commit more than once per
> exercise if solving it requires multiple complicated steps.

## Imports and paths

In [ ]:
from pathlib import Path

import pandas as pd

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = False
pd.options.plotting.backend = "plotly"

this_dir = Path()
this_dir.resolve()

## Running pytask

---
### Question 1

Run pytask in the current directory and find out what it does!


It executes two independent tasks, each producing one clean table in normal
form:

- `task_clean_election_data`: reads `Data_Elections.dta`, cleans it (as in
  session 4), and writes `bld/election_results.pkl`.
- `task_clean_alesina_data`: reads `Alesina.xlsx`, picks country, year, and
  two deficit definitions, and writes `bld/deficits.pkl`.

Find out via:

- `pytask collect --nodes`
- `pytask`
- inspecting the source code and the resulting directories

## Merging

---
### Question 2

Read in the cleaned election results and the deficits data (raw and primary
deficit by country and year). Both were produced by the pytask tasks you just
discovered.

In [ ]:

cleaned_data = pd.read_pickle(this_dir / "bld" / "election_results.pkl")
def_data = pd.read_pickle(this_dir / "bld" / "deficits.pkl")

---
### Question 3

Why do we keep election results and deficits in two separate tables instead
of merging them into one at the very start? Relate your answer to the
principles of normal forms from the screencast on "Managing data with a
complex structure".


- The two tables come from different sources, cover different sets of
  columns, and are cleaned independently. Keeping them separate, with each
  satisfying the rules for normal forms on its own, means that each cleaning
  task stays simple and testable.
- Merging early would force us to pick a merge key and a type of merge before
  we even know what analysis we want to run. On top of that, each table's
  cleaning logic would start to depend on the other table.
- Storing one table per source and merging only when a specific analysis
  requires it (here: the scatterplots below) is the workflow recommended in
  the screencast.

---
### Question 4

Your final goal is to create scatterplots with the following properties:

- x-axis: one of the three `share_votes_far_*` variables
- y-axis: one of the two deficit measures
- colours of the dots given by the NUTS region
- readable names in the legend
- focus on years with elections

To accomplish this task, you will need the following columns from the
elections data:

In [ ]:
election_cols = [
    "country",
    "year",
    "nuts_name",
    "election_type",
    "share_votes_far_right",
    "share_votes_far_left",
    "share_votes_far_any",
]

Explain why you need each of those columns! 


- `country`: needed for merging with the deficit data
- `year`: needed for merging with the deficit data
- `nuts_name`: determines the colours of the dots
- `election_type`: allows restricting the data to years with elections
- `share_votes_far_right`: one of the axes in a plot
- `share_votes_far_left`: one of the axes in a plot
- `share_votes_far_any`: one of the axes in a plot

---
### Question 5

Think about how you would go about merging the elections data with the
deficits data in order to accomplish the goal outlined above.

Focus on the following dimensions:

1. Merge key(s), i.e., the column(s) to merge on
2. Type of merge (inner, outer, left, right)


- We want to merge on country and year because this is the level of
  aggregation in the deficit data.
- We want an inner join. The election data only contains European countries;
  there is no point in keeping countries that appear in just one of the
  datasets. The same goes for years.

---
### Question 6

Check whether the contents of the merge keys that you identified are
compatible in the two datasets.

> **Hint:** One way to achieve this is to convert each of the columns'
> contents to a set and check whether their intersection is what you expect
> it to be!

In [ ]:

set(def_data["country"]).intersection(set(cleaned_data["country"]))

In [ ]:

set(def_data["year"]).intersection(set(cleaned_data["year"]))


Looks good, we can proceed.

---
### Question 7

Perform the merge operation that you identified, keeping only the columns
`election_cols` from the election data.

Assign the resulting object to a variable `final_data`.

In [ ]:

final_data = cleaned_data[election_cols].merge(
    right=def_data,
    how="inner",
    on=["country", "year"],
)

---
### Question 8

Make a scatterplot for years with elections that shows:

- the primary deficit on the x-axis
- the share of votes for any extremist parties on the y-axis
- the dots coloured by the NUTS region

In [ ]:

only_elections = final_data[final_data["election_type"].notna()]
only_elections.plot.scatter(
    x="primary_deficit",
    y="share_votes_far_any",
    color="nuts_name",
)

## Writing (py)tasks

---
### Question 9

Create a file `task_merge_data.py`, which contains a task function that reads
in the two cleaned datasets, merges them as in Question 7 (keeping only the
columns `election_cols` from the election data), and saves the resulting
DataFrame in a file `merged_data.pkl` in the `bld` directory.

Verify that this task is being run if you start pytask from the command line
and that the result is the same as the merged data from Question 7.


Reference solution for `task_merge_data.py`:

```python
from pathlib import Path

import pandas as pd

BLD = Path(__file__).parent / "bld"


def task_merge_data(
    election_results_file=BLD / "election_results.pkl",
    deficits_file=BLD / "deficits.pkl",
    produces=BLD / "merged_data.pkl",
):
    elec = pd.read_pickle(election_results_file)
    deficits = pd.read_pickle(deficits_file)
    merged = _merge_data(elec=elec, deficits=deficits)
    merged.to_pickle(produces)


def _merge_data(elec, deficits):
    election_cols = [
        "country",
        "year",
        "nuts_name",
        "election_type",
        "share_votes_far_right",
        "share_votes_far_left",
        "share_votes_far_any",
    ]
    return elec[election_cols].merge(
        right=deficits,
        how="inner",
        on=["country", "year"],
    )
```

You can check that the contents are as expected via
`pd.read_pickle(this_dir / "bld" / "merged_data.pkl").equals(final_data)`.

---
### Question 10

Create a file `task_create_scatterplots.py`, which contains a task function
that reads in the merged data and creates 6 scatterplots similar to the one
above. They should vary along what is depicted on their x-axes and y-axes:

- the two deficit measures
- the three `share_votes_far_*` variables

In all cases, dots should be coloured by the NUTS region. Give the files
suitable names.

> *Note:* Exporting static images with `plotly` needs a Chrome or Chromium
> browser, which `kaleido` drives behind the scenes. If the export fails
> because none is found, run `pixi run plotly_get_chrome` once in your
> exercise repository.


Reference solution for `task_create_scatterplots.py`:

```python
from pathlib import Path

import pandas as pd

pd.options.plotting.backend = "plotly"


BLD = Path(__file__).parent / "bld"


products = {}
for def_type in ("raw_deficit", "primary_deficit"):
    for vote_type in ("far_right", "far_left", "far_any"):
        products[(def_type, vote_type)] = (
            BLD / f"scatterplot_{def_type}_vs_{vote_type}.png"
        )


def task_create_scatterplots(
    data_file=BLD / "merged_data.pkl",
    produces=products,
):
    df = pd.read_pickle(data_file)
    # Restrict the data once and for all
    df = df[df["election_type"].notna()]
    for def_type, vote_type in produces:
        fig = df.plot.scatter(
            x=def_type, y=f"share_votes_{vote_type}", color="nuts_name"
        )
        fig.write_image(produces[(def_type, vote_type)])
```

Running `pytask` should now also produce six `.png` files in the `bld`
directory.